The purpose of this notebook is to print the movies.csv, clean up missing values, handle genres, and build a basic content-based recommendation function.

In [78]:
import pandas as pd
import pycountry

### Load data

In [79]:
path = '/Users/qianwen/Desktop/hiParis-challenge/movies.csv'
df = pd.read_csv(path, engine='python')
df.head()

,Release_Date,Title,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Poster_Url
0,2021-12-15,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,5083.954,8940,8.3,en,"Action, Adventure, Science Fiction",https://image.tmdb.org/t/p/original/1g0dhYtq4i...
1,2022-03-01,The Batman,"In his second year of fighting crime, Batman u...",3827.658,1151,8.1,en,"Crime, Mystery, Thriller",https://image.tmdb.org/t/p/original/74xTEgt7R3...
2,2022-02-25,No Exit,Stranded at a rest stop in the mountains durin...,2618.087,122,6.3,en,Thriller,https://image.tmdb.org/t/p/original/vDHsLnOWKl...
3,2021-11-24,Encanto,"The tale of an extraordinary family, the Madri...",2402.201,5076,7.7,en,"Animation, Comedy, Family, Fantasy",https://image.tmdb.org/t/p/original/4j0PNHkMr5...
4,2021-12-22,The King's Man,As a collection of history's worst tyrants and...,1895.511,1793,7.0,en,"Action, Adventure, Thriller, War",https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...


### Data preprocessing: clean missing values, check errors

In [80]:
# Handle missing values
df.isnull().sum()

Release_Date          0
Title                 9
Overview              9
Popularity           10
Vote_Count           10
Vote_Average         10
Original_Language    10
Genre                11
Poster_Url           11
dtype: int64

In [81]:
# Remove rows if all columns are empty
df = df[df.notna().any(axis=1)]

In [82]:
# Remove rows when the Title is missing
df = df[df['Title'].notna()]
df['Title'] = df['Title'].str.strip()

In [83]:
# Convert Release_Date string entries into native datetime64 objects
df['Release_Date'] = pd.to_datetime(df['Release_Date'], errors='coerce')

# assign a fallback token 0 to missing date
df['Release_Date'] = df['Release_Date'].fillna(0)

In [84]:
# clean overview
df['Overview'] = df['Overview'].str.strip()
df['Overview'] = df['Overview'].fillna('')

In [85]:
def clean_genres(genre):
    if pd.isna(genre):
        return []
    return sorted({g.strip() for g in str(genre).split(',') if g.strip()})

df['Genre'] = df['Genre'].apply(clean_genres)
df['Genre'].head()


0    [Action, Adventure, Science Fiction]
1              [Crime, Mystery, Thriller]
2                              [Thriller]
3    [Animation, Comedy, Family, Fantasy]
4      [Action, Adventure, Thriller, War]
Name: Genre, dtype: object

In [86]:
# Original language clean and convert language code to real language names

def code_to_language(code):
    if pd.isna(code) or str(code).strip() == '':
        return 'Unknown language'
    code = str(code).strip().lower()
    if code == 'cn':
        return 'Chinese'
    if len(code) != 2:
        return 'Unknown language'
    lang = pycountry.languages.get(alpha_2=code)
    return lang.name if lang else code.upper()

df['Language'] = df['Original_Language'].apply(code_to_language)
df['Language'].isna().sum()

np.int64(0)

# Save the clean dataset

In [87]:
save_path = '/Users/qianwen/Desktop/hiParis-challenge/movies_clean.csv'
df.to_csv(save_path, index=False)